# 📊 Deep Research — Pairwise Report Evaluation for the domain `NLP`

This notebook evaluates **all pairwise combinations** of the four config reports for the same research question using **YESciEval pairwise rubrics**.

### Flow
1. Read the **title** from the NLP CSV (shared by all reports, used as `question`)
2. Load **all four config reports** from the report pool
3. Enumerate all **C(4,2) = 6 config pairs** automatically
4. For each pair — run the YESciEval pairwise judge across all rubrics
5. Aggregate rubric scores → **category-level scores for A and B**
6. Display a **6-panel grouped comparison plot** per pair
7. Save JSON + CSV outputs per pair

---
**Prerequisites:** `pip install yescieval python-dotenv matplotlib numpy`

## 1 — Imports & Setup

In [ ]:
!pip install yescieval python-dotenv matplotlib -q

In [ ]:
import re, json, csv, itertools
from pathlib import Path
from typing import Dict, Tuple, List
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
from dotenv import load_dotenv
load_dotenv()
print("✅ Core imports loaded")

In [ ]:
# Pairwise rubric classes — same names as pointwise but from the pairwise subpackage
from yescieval import CustomAutoJudge, VocabularyInjector, ExampleInjector
from yescieval.rubric.pairwise.depth      import TemporalPrecision, CausalReasoning, MechanisticUnderstanding
from yescieval.rubric.pairwise.breadth    import ContextCoverage, ScopeCoverage, DimensionCoverage, ScaleCoverage, MethodCoverage
from yescieval.rubric.pairwise.rigor      import EpistemicCalibration, ExplicitUncertainty, QuantitativeEvidenceAndUncertainty
# from yescieval.rubric.pairwise.innovation import StateOfTheArtAndNovelty
# from yescieval.rubric.pairwise.gap        import GapIdentification
print("✅ YESciEval pairwise imports loaded")

## 2 — Configuration

Set the domain, model, and populate the report pool. All C(4,2) = 6 pairs are evaluated automatically.

In [ ]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────

QUESTIONS_CSV  = "your_questions_csv_path_here.csv"
OUTPUT_DIR     = "your_output_dir_here"
DOMAIN         = "nlp"              # nlp or ecology
MODEL_ID       = "your_model_id_here"
DEVICE         = "cpu"              # "cpu" | "cuda" | "mps"
HF_TOKEN       = "your_hf_token_here"
MAX_NEW_TOKENS = 2048

# ── Report pool ──────────────────────────────────────────────────────────────
# Map every config to its report file. All C(4,2) = 6 pairs will be evaluated.
REPORT_POOL = {
    'd1_b1': 'reports/1_model_orkg_d1_b1.md',
    'd1_b4': 'reports/1_model_orkg_d1_b4.md',
    'd4_b1': 'reports/1_model_orkg_d4_b1.md',
    'd4_b4': 'reports/1_model_orkg_d4_b4.md',
}

# ── All pairs are generated automatically — no manual selection needed ────────
ALL_PAIRS = list(itertools.combinations(REPORT_POOL.keys(), 2))
# → [('d1_b1','d1_b4'), ('d1_b1','d4_b1'), ('d1_b1','d4_b4'),
#    ('d1_b4','d4_b1'), ('d1_b4','d4_b4'), ('d4_b1','d4_b4')]

# Category → list of (RubricClass, weight) tuples
CATEGORIES: Dict[str, List[Tuple[type, float]]] = {
    "depth": [
        (TemporalPrecision,         1/3),
        (CausalReasoning,           1/3),
        (MechanisticUnderstanding,  1/3),
    ],
    "breadth": [
        (ContextCoverage,   1/5),
        (ScopeCoverage,     1/5),
        (DimensionCoverage, 1/5),
        (MethodCoverage,    1/5),
        (ScaleCoverage,     1/5),
    ],
    "rigor": [
        (EpistemicCalibration,               1/3),
        (ExplicitUncertainty,                1/3),
        (QuantitativeEvidenceAndUncertainty, 1/3),
    ],
    # "innovation": [
    #     (StateOfTheArtAndNovelty, 1.0),
    # ],
    # "gap": [
    #     (GapIdentification, 1.0),
    # ],
}

RATING_MIN, RATING_MAX = 1, 5

print(f"Report pool : {list(REPORT_POOL.keys())}")
print(f"Pairs to run: {len(ALL_PAIRS)} — {ALL_PAIRS}")
print(f"Model       : {MODEL_ID}")
print(f"Categories  : {list(CATEGORIES.keys())}")

## 3 — Helper Functions

In [ ]:
def parse_config(stem: str) -> str:
    """Extracts dX_bY from filename, e.g. 1_o3-mini_orkg_d1_b1 -> d1_b1."""
    m = re.search(r'd(\d+)_b(\d+)', stem)
    return f"d{m.group(1)}_b{m.group(2)}" if m else 'd1_b1'


def parse_report_number(stem: str) -> int:
    """Extracts the leading number from filename, e.g. 1_o3-mini -> 1."""
    m = re.match(r'(\d+)[_\-]', stem)
    return int(m.group(1)) if m else -1


def load_question_from_csv(csv_path: str, report_number: int) -> str:
    """
    Reads the paper title from the NLP CSV by ROW POSITION (1-based).
    The title is used as the 'question' input to YESciEval.
    All reports share the same title; row is derived from the first pool entry.
    """
    encodings_to_try = ['cp1252', 'utf-8', 'utf-8-sig', 'latin-1']
    for encoding in encodings_to_try:
        try:
            with open(csv_path, newline='', encoding=encoding) as f:
                reader = csv.DictReader(f)
                for row_num, row in enumerate(reader, start=1):
                    if row_num == report_number:
                        normalised = {k.strip().lower(): v.strip() for k, v in row.items()}
                        title = normalised.get('title', '')
                        return title if title else f'Empty title at row {report_number}'
            return f'Row {report_number} not found in CSV (file has fewer rows)'
        except UnicodeDecodeError:
            continue
    return 'Could not open CSV with any known encoding'


def parse_pairwise_judge_result(result, rubric_name: str) -> Tuple[int, int, str, str]:
    """
    Extracts (rating_a, rating_b, rationale_a, rationale_b) from a pairwise judge result.

    Expected JSON shape:
    {
        "RubricName": {
            "ResponseA": {"rating": 4, "rationale": "..."},
            "ResponseB": {"rating": 3, "rationale": "..."}
        }
    }
    """
    if isinstance(result, str):
        cleaned = re.sub(r'<think>.*?</think>', '', result, flags=re.DOTALL).strip()
        json_str = None
        depth, start = 0, None
        for i, ch in enumerate(cleaned):
            if ch == '{':
                if depth == 0: start = i
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0 and start is not None:
                    json_str = cleaned[start:i+1]
                    break
        if json_str:
            try:
                result = json.loads(json_str)
            except json.JSONDecodeError:
                pass
        if isinstance(result, str):
            return 0, 0, '', result

    if isinstance(result, dict):
        inner = result.get(rubric_name) or result.get(rubric_name.lower())
        if inner is None and len(result) == 1:
            inner = next(iter(result.values()))
        if isinstance(inner, dict):
            resp_a = inner.get('ResponseA', {})
            resp_b = inner.get('ResponseB', {})
            return (
                int(resp_a.get('rating',    0)),
                int(resp_b.get('rating',    0)),
                str(resp_a.get('rationale', '')),
                str(resp_b.get('rationale', '')),
            )
        if 'ResponseA' in result:
            resp_a = result['ResponseA']
            resp_b = result.get('ResponseB', {})
            return (
                int(resp_a.get('rating',    0)),
                int(resp_b.get('rating',    0)),
                str(resp_a.get('rationale', '')),
                str(resp_b.get('rationale', '')),
            )
        return 0, 0, '', str(result)

    if hasattr(result, 'ResponseA') and hasattr(result, 'ResponseB'):
        resp_a = result.ResponseA
        resp_b = result.ResponseB
        return (
            int(getattr(resp_a, 'rating',    0)),
            int(getattr(resp_b, 'rating',    0)),
            str(getattr(resp_a, 'rationale', '')),
            str(getattr(resp_b, 'rationale', '')),
        )
    return 0, 0, '', str(result)


def compute_category_score(rubric_scores: List[Tuple[float, float]]) -> float:
    """
    Weighted mean of (score, weight) pairs, normalised from 1-5 to 0.0-1.0.
    Formula: (weighted_mean - 1) / (5 - 1)
    """
    total_w = sum(w for _, w in rubric_scores)
    if not total_w:
        return 0.0
    weighted_mean = sum(s * w for s, w in rubric_scores) / total_w
    return round((weighted_mean - 1) / (RATING_MAX - RATING_MIN), 4)


def run_pair(cfg_a, cfg_b, report_a_md, report_b_md, question, judge):
    """
    Runs the full pairwise evaluation for one (cfg_a, cfg_b) combination.
    Returns (rubric_raw, category_scores, overall_a, overall_b).
    """
    rubric_raw:      Dict[str, dict]             = {}
    category_scores: Dict[str, Dict[str, float]] = {}

    for cat_name, rubric_list in CATEGORIES.items():
        print(f"  📂 {cat_name.upper()}")
        cat_rubric_a: List[Tuple[float, float]] = []
        cat_rubric_b: List[Tuple[float, float]] = []

        for RubricClass, weight in rubric_list:
            rname = RubricClass.__name__
            print(f"     🔎 [{rname}] (w={weight:.3f}) ...")
            rubric = RubricClass(
                papers={},
                question=question,
                answer_a=report_a_md,
                answer_b=report_b_md,
                domain=DOMAIN,
                vocabulary=VocabularyInjector(),
                example=ExampleInjector(),
            )
            raw = judge.judge(rubric=rubric, max_new_tokens=MAX_NEW_TOKENS)
            rating_a, rating_b, rat_a, rat_b = parse_pairwise_judge_result(raw, rname)
            rubric_raw[rname] = {
                'ResponseA': {'rating': rating_a, 'rationale': rat_a},
                'ResponseB': {'rating': rating_b, 'rationale': rat_b},
                'weight_in': {cat_name: weight},
            }
            print(f"        ResponseA={rating_a}/5  |  ResponseB={rating_b}/5")
            cat_rubric_a.append((float(rating_a), weight))
            cat_rubric_b.append((float(rating_b), weight))

        score_a = compute_category_score(cat_rubric_a)
        score_b = compute_category_score(cat_rubric_b)
        category_scores[cat_name] = {'a': score_a, 'b': score_b}
        print(f"     ✅ {cat_name}  {cfg_a}={score_a:.2f}  {cfg_b}={score_b:.2f}")

    overall_a = round(sum(v['a'] for v in category_scores.values()) / len(category_scores), 4)
    overall_b = round(sum(v['b'] for v in category_scores.values()) / len(category_scores), 4)
    return rubric_raw, category_scores, overall_a, overall_b

print("✅ Helpers defined")

## 4 — Load All Reports & Read Title from CSV

The **title** column from the NLP CSV is used as the `question` input to YESciEval.

In [ ]:
# Load all reports from the pool upfront
loaded_reports: Dict[str, str] = {}
for cfg, path in REPORT_POOL.items():
    p = Path(path).resolve()
    if not p.exists():
        raise FileNotFoundError(f'Report not found for {cfg}: {p}')
    loaded_reports[cfg] = p.read_text(encoding='utf-8', errors='ignore')
    print(f"✅ {cfg}: {p.name}  ({len(loaded_reports[cfg]):,} chars)")

# Derive the question row from the first pool entry's filename
first_stem = Path(next(iter(REPORT_POOL.values()))).stem
report_num = parse_report_number(first_stem)
question   = load_question_from_csv(QUESTIONS_CSV, report_num)  # NLP: reads 'title' column

# Extract engine string for plot titles (reused across all pairs)
engine_m   = re.search(r'\d+_(.*?)_orkg', first_stem)
engine_str = engine_m.group(1) if engine_m else first_stem

print(f"\nReport Number    : {report_num}")
print(f"Question (title) : {question}")

## 5 — Initialise the YESciEval Judge

In [ ]:
print(f"⏳ Loading {MODEL_ID} on {DEVICE} ...")
judge = CustomAutoJudge()
judge.from_pretrained(model_id=MODEL_ID, device=DEVICE, token=HF_TOKEN or None)
print(f"✅ Judge ready")

## 6 — Run All Pairwise Evaluations

Iterates over all **C(4,2) = 6 config pairs** automatically. For each pair,
every rubric scores `answer_a` and `answer_b` independently via `ResponseA` /
`ResponseB`. Results are collected in `all_pair_results` keyed by `cfg_a_vs_cfg_b`.

In [ ]:
all_pair_results: Dict[str, dict] = {}

for cfg_a, cfg_b in ALL_PAIRS:
    pair_key = f'{cfg_a}_vs_{cfg_b}'
    print(f"\n{'='*60}")
    print(f"🔁 Pair {ALL_PAIRS.index((cfg_a, cfg_b)) + 1}/{len(ALL_PAIRS)}: {cfg_a}  vs  {cfg_b}")
    print(f"{'='*60}")

    rubric_raw, category_scores, overall_a, overall_b = run_pair(
        cfg_a, cfg_b,
        loaded_reports[cfg_a],
        loaded_reports[cfg_b],
        question,
        judge,
    )

    all_pair_results[pair_key] = {
        'config_a':        cfg_a,
        'config_b':        cfg_b,
        'rubric_scores':   rubric_raw,
        'category_scores': category_scores,
        'overall_a':       overall_a,
        'overall_b':       overall_b,
    }
    print(f"\n  🏁 {pair_key}  overall  {cfg_a}={overall_a:.2f}  {cfg_b}={overall_b:.2f}  (0-1 scale)")

print(f"\n✅ All {len(ALL_PAIRS)} pairs evaluated.")

## 7 — Quality Dimensions Plots

One 6-panel grouped bar chart per pair. Blue = Config A, Red = Config B.
Only the two configs in each pair have non-zero bars so their relative positions
on the shared x-axis remain visually comparable across all plots.

In [ ]:
PANEL_TITLES = {
    'depth':      'Research Depth Score',
    'breadth':    'Research Breadth Score',
    'rigor':      'Scientific Rigor Score',
    'innovation': 'Innovation Score',
    'gap':        'Research Gap Score',
    'overall':    'Overall Quality Score',
}

ALL_CONFIGS = ['d1_b1', 'd1_b4', 'd4_b1', 'd4_b4']
xs    = np.arange(len(ALL_CONFIGS))
BAR_W = 0.22

pair_figs: Dict[str, plt.Figure] = {}   # store figures for saving in Cell 9

for pair_key, res in all_pair_results.items():
    cfg_a = res['config_a']
    cfg_b = res['config_b']
    category_scores = res['category_scores']
    overall_a = res['overall_a']
    overall_b = res['overall_b']

    plot_scores = {k: category_scores.get(k, {'a': 0.0, 'b': 0.0})
                  for k in list(PANEL_TITLES.keys())[:-1]}
    plot_scores['overall'] = {'a': overall_a, 'b': overall_b}

    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.ravel()

    for i, (cat, panel_title) in enumerate(PANEL_TITLES.items()):
        ax   = axes[i]
        vals = plot_scores[cat]

        bar_a = [vals['a'] if cfg == cfg_a else 0.0 for cfg in ALL_CONFIGS]
        bar_b = [vals['b'] if cfg == cfg_b else 0.0 for cfg in ALL_CONFIGS]

        ax.bar(xs - BAR_W/2, bar_a, width=BAR_W, color='#2980b9', label=cfg_a)
        ax.bar(xs + BAR_W/2, bar_b, width=BAR_W, color='#e74c3c', label=cfg_b)

        ax.set_title(panel_title, fontsize=12)
        ax.set_xticks(list(xs))
        ax.set_xticklabels(ALL_CONFIGS, fontsize=10)
        ax.set_ylim(0, 1.1)
        ax.set_ylabel('Score (0..1)', fontsize=9)
        ax.legend(fontsize=8)

        for x, va, vb in zip(xs, bar_a, bar_b):
            if va: ax.text(x - BAR_W/2, va + 0.02, f'{va:.2f}', ha='center', va='bottom', fontsize=8)
            if vb: ax.text(x + BAR_W/2, vb + 0.02, f'{vb:.2f}', ha='center', va='bottom', fontsize=8)

    fig.suptitle(
        f'Pairwise Quality Comparison — {DOMAIN} / {engine_str}\n{cfg_a} vs {cfg_b}',
        fontsize=15
    )
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    pair_figs[pair_key] = fig
    plt.show()
    print(f"📊 {pair_key} rendered")

## 8 — Final Evaluation Output (JSON)

Prints the full results dict covering all 6 pairs.

In [ ]:
final_output = {
    'domain':        DOMAIN,
    'report_number': report_num,
    'question':      question,   # NLP: paper title
    'model':         MODEL_ID,
    'pairs':         all_pair_results,   # keyed by 'cfg_a_vs_cfg_b'
}
display(Markdown('---\n### 🏁 Final Output\n---'))
print(json.dumps(final_output, indent=2, ensure_ascii=False))

## 9 — Save Outputs to Disk

One JSON, one CSV row, and one plot image are saved **per pair**.

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# ── CSV: one row per pair, all pairs in a single file ────────────────────────
csv_path = out_dir / f'pairwise_all_scores_{DOMAIN}.csv'
csv_rows = []
for pair_key, res in all_pair_results.items():
    row = {
        'pair':     pair_key,
        'config_a': res['config_a'],
        'config_b': res['config_b'],
        'title':    question,   # NLP: paper title used as question
    }
    for cat, scores in res['category_scores'].items():
        row[f'{cat}_score_a'] = round(scores['a'], 4)
        row[f'{cat}_score_b'] = round(scores['b'], 4)
    row['overall_score_a'] = round(res['overall_a'], 4)
    row['overall_score_b'] = round(res['overall_b'], 4)
    csv_rows.append(row)

with csv_path.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(csv_rows[0].keys()))
    w.writeheader()
    w.writerows(csv_rows)
print(f"💾 CSV   → {csv_path}")

# ── JSON + plot: one file per pair ───────────────────────────────────────────
for pair_key, res in all_pair_results.items():
    pair_output = {
        'domain':          DOMAIN,
        'report_number':   report_num,
        'question':        question,   # NLP: paper title
        'model':           MODEL_ID,
        **res,
    }
    json_path = out_dir / f'{pair_key}_scores.json'
    json_path.write_text(json.dumps(pair_output, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f"💾 JSON  → {json_path}")

    fig_path = out_dir / f'{pair_key}_quality_dimensions.png'
    pair_figs[pair_key].savefig(fig_path, dpi=200, bbox_inches='tight')
    print(f"💾 Plot  → {fig_path}")